# Language processing with Groovy

Five NLP tasks using [Apache OpenNLP](https://opennlp.apache.org/) (Apache-2.0
throughout), ported from the BeakerX-era `LanguageProcessing.ipynb` in
[groovy-data-science](https://github.com/paulk-asert/groovy-data-science):
language detection, sentence detection, named-entity recognition,
parts-of-speech tagging, and sentiment analysis.

Model handling is nicer than in the BeakerX era: OpenNLP now publishes many
models **on Maven Central**, so language-detection and POS models arrive via
`@Grab` and load straight off the classpath. Two classic models with no
modern equivalent (abbreviation-aware sentence detection, and the 1.5-era
named-entity models) are downloaded once into a `models/` directory beside
this notebook and cached. The sentiment model is trained locally from the
included `rt-polarity` movie-review dataset (see `rt-polarity-source.txt`).

In [1]:
@Grab('org.apache.opennlp:opennlp-tools:2.5.11')
@Grab('org.apache.opennlp:opennlp-models-langdetect:1.3.0')
@Grab('org.apache.opennlp:opennlp-models-pos-en:1.3.0')
@Grab('org.apache.opennlp:opennlp-models-tokenizer-en:1.3.0')
import opennlp.tools.langdetect.*
cl = this.class.classLoader
modelsDir = new File('models')
modelsDir.mkdirs()
cached = { name ->
    def f = new File(modelsDir, name)
    if (!f.exists()) f.bytes = new URL("https://opennlp.sourceforge.net/models-1.5/$name").bytes
    f
}
'deps ready'

deps ready

## Language detection

The detector model ships as a Maven Central artifact — grabbed above, loaded
from the classpath here:

In [2]:
import opennlp.tools.langdetect.*
detector = new LanguageDetectorME(new LanguageDetectorModel(cl.getResourceAsStream('langdetect-183.bin')))
['Bienvenido a Madrid', 'Bienvenue à Paris',
 'Добре дошли в София', 'Velkommen til København'].collect {
    def best = detector.predictLanguage(it)
    [text: it, language: best.lang, confidence: (best.confidence as double).round(3)]
}

text,language,confidence
Bienvenido a Madrid,spa,0.017
Bienvenue à Paris,fra,0.019
Добре дошли в София,bul,0.025
Velkommen til København,dan,0.025


## Sentence detection

This text contains 28 full stops but only 4 sentences — the rest belong to
abbreviations and initials. The classic abbreviation-aware English model
handles it correctly (the newer UD-trained sentence model on Central splits
this text into 6, so we cache the classic one):

In [3]:
import opennlp.tools.sentdetect.*
text = '''
The most referenced scientific paper of all time is "Protein measurement with the
Folin phenol reagent" by Lowry, O. H., Rosebrough, N. J., Farr, A. L. & Randall,
R. J. and was published in the J. BioChem. in 1951. It describes a method for
measuring the amount of protein (even as small as 0.2 γ, were γ is the specific
weight) in solutions and has been cited over 300,000 times and can be found here:
https://www.jbc.org/content/193/1/265.full.pdf. Dr. Lowry completed
two doctoral degrees under an M.D.-Ph.D. program from the University of Chicago
before moving to Harvard under A. Baird Hastings. He was also the H.O.D of
Pharmacology at Washington University in St. Louis for 29 years.
'''
sentenceDetector = new SentenceDetectorME(new SentenceModel(cached('en-sent.bin')))
sentences = sentenceDetector.sentDetect(text)
assert text.count('.') == 28 && sentences.size() == 4
(0..<sentences.size()).collect { [n: it + 1, sentence: sentences[it].replaceAll('\\s+', ' ')] }

n,sentence
1,"The most referenced scientific paper of all time is ""Protein measurement with the Folin phenol reagent"" by Lowry, O. H., Rosebrough, N. J., Farr, A. L. & Randall, R. J. and was published in the J. BioChem. in 1951."
2,"It describes a method for measuring the amount of protein (even as small as 0.2 γ, were γ is the specific weight) in solutions and has been cited over 300,000 times and can be found here: https://www.jbc.org/content/193/1/265.full.pdf."
3,Dr. Lowry completed two doctoral degrees under an M.D.-Ph.D. program from the University of Chicago before moving to Harvard under A. Baird Hastings.
4,He was also the H.O.D of Pharmacology at Washington University in St. Louis for 29 years.


## Named-entity recognition

Five 1.5-era English models (person, money, date, time, location — cached on
first use). Note the disambiguation in the results: *Daniel Sun* vs *Daniel*
+ *Sun.*, and *May*/*June* as people in one sentence but dates in another:

In [4]:
import opennlp.tools.namefind.*
import opennlp.tools.tokenize.SimpleTokenizer
findersByType = ['person', 'money', 'date', 'time', 'location'].collectEntries {
    [it, new NameFinderME(new TokenNameFinderModel(cached("en-ner-${it}.bin")))]
}
nerSentences = [
    "A commit by Daniel Sun on December 6, 2020 improved Groovy 4's language integrated query.",
    "A commit by Daniel on Sun. December 6, 2020 improved Groovy 4's language integrated query.",
    'The Groovy in Action book by Dierk Koenig et. al. is a bargain at $50, or indeed any price.',
    'The conference wrapped up yesterday at 5:30 p.m. in Copenhagen, Denmark.',
    'I saw Ms. May Smith waving to June Jones.',
    'The parcel was passed from May to June.'
]
simple = SimpleTokenizer.INSTANCE
hits = []
nerSentences.eachWithIndex { sentence, si ->
    String[] tokens = simple.tokenize(sentence)
    findersByType.each { type, finder ->
        finder.find(tokens).each { span ->
            hits << [sentence: si + 1, entity: tokens[span.start..<span.end].join(' '),
                     type: type, probability: (span.prob as double).round(2)]
        }
    }
}
hits.sort { [it.sentence, it.type] }

sentence,entity,type,probability
1,Daniel Sun,person,0.76
2,Daniel,person,0.87
3,Dierk Koenig,person,0.91
5,May Smith,person,0.97
5,June Jones,person,0.78
1,"December 6 , 2020",date,0.83
2,"December 6 , 2020",date,0.89
4,yesterday,date,0.99
6,May to June,date,0.78
4,5 : 30 p . m .,time,0.99


## Parts-of-speech tagging

The modern POS and tokenizer models (from Central, classpath-loaded) use the
[Universal Dependencies tagset](https://universaldependencies.org/u/pos/):
`NOUN`, `PROPN` (proper noun), `VERB`, `ADJ`, `NUM`, `CCONJ` (coordinating
conjunction), `PUNCT`, and friends:

In [5]:
import opennlp.tools.postag.*
import opennlp.tools.tokenize.*
tokenizer = new TokenizerME(new TokenizerModel(cl.getResourceAsStream('opennlp-en-ud-ewt-tokens-1.3-2.5.4.bin')))
posTagger = new POSTaggerME(new POSModel(cl.getResourceAsStream('opennlp-en-ud-ewt-pos-1.3-2.5.4.bin')))
['Paul has two sisters, Maree and Christine.',
 'His bark was much worse than his bite',
 'Turn on the lights to the master bedroom',
 "Light 'em all up",
 'Make it dark downstairs'].collect { s ->
    def toks = tokenizer.tokenize(s)
    [sentence: s, tagged: [toks, posTagger.tag(toks)].transpose().collect { t, tag -> "$t/$tag" }.join(' ')]
}

sentence,tagged
"Paul has two sisters, Maree and Christine.","Paul/PROPN has/VERB two/NUM sisters/NOUN ,/PUNCT Maree/PROPN and/CCONJ Christine/PROPN ./PUNCT"
His bark was much worse than his bite,His/PRON bark/NOUN was/AUX much/ADV worse/ADJ than/ADP his/PRON bite/NOUN
Turn on the lights to the master bedroom,Turn/VERB on/ADP the/DET lights/NOUN to/ADP the/DET master/NOUN bedroom/NOUN
Light 'em all up,Light/NOUN 'em/PUNCT all/DET up/ADP
Make it dark downstairs,Make/VERB it/PRON dark/ADJ downstairs/NOUN


## Sentiment analysis

Train a document categorizer on the included `rt-polarity` movie-review
dataset (5,331 positive + 5,331 negative snippets). Training takes well under
a second; progress lines print as the maxent trainer iterates:

In [6]:
import opennlp.tools.doccat.*
import opennlp.tools.util.*
start = System.currentTimeMillis()
training = [positive: 'rt-polarity.pos', negative: 'rt-polarity.neg'].collect { cat, file ->
    new File(file).readLines('ISO-8859-1').collect { "$cat $it".toString() }
}.sum()
sentimentModel = DocumentCategorizerME.train('en',
    new DocumentSampleStream(new CollectionObjectStream(training)),
    new TrainingParameters(), new DoccatFactory())
"trained on ${training.size()} reviews in ${System.currentTimeMillis() - start}ms"

trained on 10661 reviews in 445ms

In [7]:
categorizer = new DocumentCategorizerME(sentimentModel)
['OpenNLP is fantastic!', 'Groovy is great fun!', 'Math can be hard!'].collect { s ->
    def probs = categorizer.categorize(s.split('[ !]'))
    def best = categorizer.getBestCategory(probs)
    [sentence: s, sentiment: best, confidence: probs[categorizer.getIndex(best)].round(2)]
}

sentence,sentiment,confidence
OpenNLP is fantastic!,positive,0.64
Groovy is great fun!,positive,0.74
Math can be hard!,negative,0.61
